<a href="https://colab.research.google.com/github/421110016-a11y/metro-cdmx-dfs-bfs/blob/main/Hill_Climbing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Implementación del algoritmo Hill Climbing

## Sistemas Inteligentes

**Autor:** Luis Ángel Ciprés Flores

### Objetivo

Implementar el algoritmo de búsqueda local **Hill Climbing (Ascenso de Colina)** utilizando las clases `Problem` y `Node` desarrolladas previamente en clase.

El algoritmo será aplicado sobre un problema representado mediante un grafo, utilizando una función heurística para seleccionar, en cada iteración, el estado vecino que represente una mejora respecto al estado actual.

### Hill Climbing

Hill Climbing es un algoritmo de búsqueda local que comienza en un estado inicial y analiza los estados vecinos disponibles.

En cada iteración selecciona el vecino que presenta el mejor valor de la función heurística. El proceso continúa mientras exista una mejora respecto al estado actual.

A diferencia de otros algoritmos de búsqueda, Hill Climbing no mantiene todas las rutas posibles, sino que avanza tomando decisiones locales.

Una de sus principales limitaciones es que puede detenerse antes de alcanzar la solución global debido a mínimos o máximos locales y mesetas.

## **1. Clase `Problem**`

La clase `Problem` representa la estructura general de un problema de búsqueda.

En ella se definen los elementos fundamentales del problema:

- `initial`: estado inicial.
- `goal`: estado objetivo.
- `actions()`: acciones disponibles desde un estado.
- `result()`: estado resultante después de ejecutar una acción.
- `is_goal()`: determina si un estado corresponde al objetivo.
- `action_cost()`: costo asociado a una transición.
- `h()`: función heurística utilizada para estimar la conveniencia de un estado.

Las funciones `actions()` y `result()` se dejan sin implementación general porque su comportamiento dependerá del problema específico que se quiera resolver.

In [2]:
# Clase abstracta
class Problem:
    def __init__(self, initial, goal):
        self.initial = initial  # Estado inicial
        self.goal = goal        # Meta

    def actions(self, state):
        raise NotImplementedError

    # Función de transición
    def result(self, state, action):
        raise NotImplementedError

    # Función de desempeño
    def is_goal(self, state):
        return self.goal == state

    def action_cost(self, state1, action, state2):
        return 1

    def h(self, state):
        return 0

## 2. Clase `GraphAStartProblem`

La clase `GraphAStartProblem` hereda de la clase `Problem` y permite representar un problema de búsqueda mediante un grafo.

Esta clase utiliza:

- `graph`: estructura que contiene los estados y sus conexiones.
- `actions()`: obtiene los estados vecinos disponibles desde el estado actual.
- `result()`: devuelve el estado al que se llega después de realizar una acción.
- `action_cost()`: obtiene el costo de desplazarse entre dos estados.
- `h()`: devuelve el valor de la función heurística correspondiente a cada estado.

Para este ejercicio se conservará la misma estructura utilizada en clase. La función heurística será posteriormente utilizada por el algoritmo Hill Climbing para decidir cuál de los estados vecinos representa la mejor alternativa.

In [3]:
class GraphAStartProblem(Problem):
    def __init__(self, initial, goal, graph):
        super().__init__(initial, goal)
        self.graph = graph

    def actions(self, state):
        lista = []
        for key in self.graph[state].keys():
            lista.append(key)
        return lista

    # Función de transición
    def result(self, state, action):
        return action

    def action_cost(self, state1, action, state2):
        return self.graph[state1][state2]

    def h(self, state):
        return straight_line_distance[state]

In [4]:
def h(self, state):
    return straight_line_distance[state]

## 3. Clase `Node`

La clase `Node` representa un nodo dentro del proceso de búsqueda.

Cada nodo almacena información sobre el estado actual y permite conservar la trayectoria seguida desde el estado inicial.

Sus principales atributos son:

- `state`: estado representado por el nodo.
- `parent`: nodo desde el cual se llegó al estado actual.
- `action`: acción utilizada para llegar al nodo.
- `path_cost`: costo acumulado desde el estado inicial.

El método `path()` permite reconstruir la ruta desde el estado inicial hasta el nodo actual.

El método `expand()` genera los nodos sucesores a partir de las acciones disponibles en el problema.

Finalmente, `child_node()` crea un nuevo nodo calculando el estado siguiente y el costo acumulado de la trayectoria.

In [5]:
class Node:
    def __init__(self, state, parent = None, action = None, path_cost = 0):
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost

    def path(self):
        lista_path = []
        node = self
        while node:
            lista_path.append(node.state)
            node = node.parent
        return lista_path[::-1]

    def expand(self, problem):
        lista = []
        for action in problem.actions(self.state):
            lista.append(self.child_node(problem, action))
        return lista

    def child_node(self, problem, action):
        next_state = problem.result(self.state, action)
        step_cost = problem.action_cost(self.state, action, next_state)
        return Node(next_state, self, action, self.path_cost + step_cost)

## 4. Grafo de Rumania

Para probar el algoritmo Hill Climbing se utilizará el problema clásico del mapa de Rumania.

Cada ciudad representa un **estado o nodo** del problema, mientras que las conexiones entre ciudades representan las posibles acciones disponibles.

El grafo contiene también el costo real de desplazamiento entre ciudades, expresado como distancia.

Por ejemplo, desde `Arad` existen tres posibles movimientos:

- Arad → Zerind: 75
- Arad → Sibiu: 140
- Arad → Timisoara: 118

Aunque estos costos forman parte del problema, Hill Climbing tomará principalmente sus decisiones utilizando la función heurística `h(n)`.

In [6]:
romania = {
    'Arad': {'Zerind': 75, 'Sibiu': 140, 'Timisoara': 118},
    'Zerind': {'Arad': 75, 'Oradea': 71},
    'Oradea': {'Zerind': 71, 'Sibiu': 151},
    'Sibiu': {'Arad': 140, 'Oradea': 151, 'Fagaras': 99, 'Rimnicu Vilcea': 80},
    'Timisoara': {'Arad': 118, 'Lugoj': 111},
    'Lugoj': {'Timisoara': 111, 'Mehadia': 70},
    'Mehadia': {'Lugoj': 70, 'Dobreta': 75},
    'Dobreta': {'Mehadia': 75, 'Craiova': 120},
    'Craiova': {'Dobreta': 120, 'Rimnicu Vilcea': 146, 'Pitesti': 138},
    'Rimnicu Vilcea': {'Sibiu': 80, 'Craiova': 146, 'Pitesti': 97},
    'Fagaras': {'Sibiu': 99, 'Bucarest': 211},
    'Pitesti': {'Rimnicu Vilcea': 97, 'Craiova': 138, 'Bucarest': 101},
    'Bucarest': {'Fagaras': 211, 'Pitesti': 101, 'Giurgiu': 90, 'Urziceni': 85},
    'Giurgiu': {'Bucarest': 90},
    'Urziceni': {'Bucarest': 85, 'Hirsova': 98, 'Vaslui': 142},
    'Hirsova': {'Urziceni': 98, 'Eforie': 86},
    'Eforie': {'Hirsova': 86},
    'Vaslui': {'Urziceni': 142, 'Iasi': 92},
    'Iasi': {'Vaslui': 92, 'Neamt': 87},
    'Neamt': {'Iasi': 87},
}

## 5. Función heurística

Hill Climbing necesita una función que permita evaluar qué tan conveniente es cada estado.

En este problema se utilizará como heurística la **distancia en línea recta desde cada ciudad hasta Bucarest**.

La función heurística se representa mediante:

\[
h(n)
\]

donde `n` representa una ciudad.

Como el objetivo es llegar a Bucarest, un valor menor de `h(n)` representa una ciudad que se encuentra, según la estimación heurística, más cerca del objetivo.

Por ejemplo:

- Arad: h(n) = 366
- Sibiu: h(n) = 253
- Fagaras: h(n) = 178
- Bucarest: h(n) = 0

Por lo tanto, en este problema Hill Climbing buscará reducir progresivamente el valor de la heurística.

In [7]:
# Distancias lineales de cada ciudad a Bucarest
straight_line_distance = {
    'Arad': 366,
    'Bucarest': 0,
    'Craiova': 160,
    'Dobreta': 242,
    'Eforie': 161,
    'Fagaras': 178,
    'Giurgiu': 77,
    'Hirsova': 151,
    'Iasi': 226,
    'Lugoj': 244,
    'Mehadia': 241,
    'Neamt': 234,
    'Oradea': 380,
    'Pitesti': 98,
    'Rimnicu Vilcea': 193,
    'Sibiu': 253,
    'Timisoara': 329,
    'Urziceni': 80,
    'Vaslui': 199,
    'Zerind': 374,
}

## 6. Algoritmo Hill Climbing

El algoritmo **Hill Climbing** es un método de búsqueda local que parte de un estado inicial y evalúa los estados vecinos disponibles.

En cada iteración, el algoritmo selecciona el vecino que presenta el mejor valor de la función heurística.

En este problema, la heurística representa la distancia estimada hasta Bucarest, por lo que un valor menor de `h(n)` representa una mejor alternativa.

El procedimiento utilizado es:

1. Crear un nodo con el estado inicial.
2. Generar los nodos vecinos mediante `expand()`.
3. Evaluar la heurística `h(n)` de cada vecino.
4. Seleccionar el vecino con el menor valor heurístico.
5. Comparar el vecino seleccionado con el estado actual.
6. Si el vecino mejora la heurística, avanzar hacia él.
7. Si ningún vecino representa una mejora, detener la búsqueda.

Hill Climbing toma decisiones locales y no mantiene una frontera con todas las rutas posibles. Por esta razón, puede encontrar una solución rápidamente, pero no garantiza alcanzar el óptimo global en todos los problemas.

In [8]:
def hill_climbing(problem):

    # Crear el nodo inicial
    current = Node(problem.initial)

    while True:

        # Generar los vecinos del nodo actual
        neighbors = current.expand(problem)

        # Si no existen vecinos, termina la búsqueda
        if not neighbors:
            return current

        # Seleccionar el vecino con menor heurística
        neighbor = min(
            neighbors,
            key=lambda node: problem.h(node.state)
        )

        # Si el mejor vecino no mejora al estado actual,
        # Hill Climbing se detiene
        if problem.h(neighbor.state) >= problem.h(current.state):
            return current

        # Avanzar al mejor vecino
        current = neighbor

## 7. Ejecución de Hill Climbing

Para probar el algoritmo se define un problema cuyo estado inicial es **Arad** y cuyo estado objetivo es **Bucarest**.

Hill Climbing utilizará la distancia en línea recta a Bucarest como función heurística para seleccionar, en cada iteración, el vecino que represente una mejora respecto al estado actual.

In [9]:
problem = GraphAStartProblem(
    "Arad",
    "Bucarest",
    romania
)

node = hill_climbing(problem)

node.path()

['Arad', 'Sibiu', 'Fagaras', 'Bucarest']

## 8. Interpretación del resultado

El algoritmo Hill Climbing encontró la siguiente secuencia de estados:

**Arad → Sibiu → Fagaras → Bucarest**

La búsqueda comienza en **Arad**, cuya heurística es `h(n) = 366`.
A partir de este estado, el algoritmo evalúa las ciudades vecinas y selecciona
aquella que presenta el menor valor heurístico.

El proceso de selección fue:

| Estado actual | h(n) | Mejor vecino | h(n) del vecino |
|---|---:|---|---:|
| Arad | 366 | Sibiu | 253 |
| Sibiu | 253 | Fagaras | 178 |
| Fagaras | 178 | Bucarest | 0 |

En cada movimiento se observa una disminución de la función heurística:

**366 → 253 → 178 → 0**

Esto indica que en cada iteración Hill Climbing seleccionó un estado que,
de acuerdo con la heurística, se encontraba más cerca del objetivo.

Finalmente, el algoritmo alcanzó **Bucarest**, donde `h(n) = 0`, por lo que
se alcanzó el estado objetivo.

Este resultado muestra el comportamiento característico de Hill Climbing:
el algoritmo toma decisiones locales seleccionando en cada paso el vecino
que representa la mayor mejora inmediata.

Sin embargo, que Hill Climbing haya alcanzado Bucarest en este ejemplo no
significa que siempre encuentre la solución óptima. En otros problemas puede
detenerse en un óptimo local o en una meseta si ninguno de los estados vecinos
representa una mejora.